# ModernBERT KNN & Ensemble Trainer (Refactored)

This notebook is a refactored version of the training script. It supports both KNN-based training and Ensemble training for hate speech detection. It uses a structured, object-oriented approach.

In [1]:
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_recall_fscore_support, confusion_matrix, balanced_accuracy_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    EvalPrediction,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from transformers import pipeline
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import Any, Dict, List, Union, Optional
import warnings
import dataHandler as dh # Ensure dataHandler.py is in the same directory

# CheckList Imports
try:
    from checklist.pred_wrapper import PredictorWrapper
    from checklist.test_types import MFT
except ImportError:
    print("CheckList library not found. Please install it using `pip install checklist`.")

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

## 1. Setup and Configuration

In [2]:
def setup_device():
    """Checks compute device availability and sets up the device."""
    print("="*60)
    print("CHECKING COMPUTE DEVICE STATUS")
    print("="*60)
    
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"✓ CUDA available: True")
        print(f"✓ CUDA device count: {torch.cuda.device_count()}")
        print(f"✓ Current device: {torch.cuda.current_device()}")
        print(f"✓ Device name: {torch.cuda.get_device_name(0)}")
        try:
            torch.cuda.empty_cache()
            print(f"✓ CUDA cache cleared successfully")
        except Exception as e:
            print(f"⚠ Warning clearing CUDA cache: {e}")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
         device = torch.device("mps")
         print("✓ MPS (Apple Silicon) available: True")
    else:
        device = torch.device("cpu")
        print("✓ CUDA not available, using CPU")
        
    return device

device = setup_device()
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

CHECKING COMPUTE DEVICE STATUS
✓ CUDA available: True
✓ CUDA device count: 1
✓ Current device: 0
✓ Device name: NVIDIA GeForce RTX 4070
✓ CUDA cache cleared successfully


In [3]:
@dataclass
class Config:
    """Configuration for training and evaluation."""
    # Paths
    knn_datasets: Dict[str, str] = None
    output_dir_base: str = "models_knn_tox_refactored"
    
    # Model
    #model_id: str = "microsoft/deberta-v3-base"
    #model_id: str = "tomh/toxigen_roberta"
    model_id: str = "answerdotai/ModernBERT-base"
    num_labels: int = 2
    labels: List[str] = None
    mode = "knnratio"
    
    # Ensemble Settings
    use_ensemble: bool = True
    identity_terms: List[str] = field(default_factory=lambda: [
        'asian', 'black', 'chinese', 'jewish', 'latino', 'lgbtq', 
        'mental_dis', 'mexican', 'middle_east', 'muslim', 
        'native_american', 'physical_dis', 'women'
    ])
    
    # Data Processing
    max_length: int = 140
    test_size: float = 0.2
    random_state: int = 42
    
    # Density Settings
    density_column: str = "density_ratio"
    normalize_density: bool = False
    use_density: bool = False
    remove_outliers: bool = False
    outlier_lower_q: float = 0.0005
    outlier_upper_q: float = 0.9995
    
    # Training
    batch_size: int = 32
    learning_rate: float = 2e-5
    num_epochs: int = 3
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    logging_steps: int = 50
    eval_steps: int = 1000
    save_steps: int = 1000
    save_limit: int = 3
    
    def __post_init__(self):
        if self.knn_datasets is None:
            self.knn_datasets = {
                #'k5': 'selected_knn_new/samples_knn_k5_percent100.csv',
                #'k100': 'selected_knn_new/samples_knn_k100_percent100.csv',
                #'k1000': 'selected_knn_new/samples_knn_k1000_percent100.csv',
                'k5_ratio': 'select_knn_complete_2/samplex_knnratio_k5.csv',
                'k100_ratio': 'select_knn_complete_2/samplex_knnratio_k100.csv',
                'k1000_ratio': 'select_knn_complete_2/samplex_knnratio_k1000.csv',
            }
        if self.labels is None:
            self.labels = ["no hate", "hate"]
            
    @property
    def label2id(self):
        return {l: str(i) for i, l in enumerate(self.labels)}
        
    @property
    def id2label(self):
        return {str(i): l for i, l in enumerate(self.labels)}
        
    @property
    def output_dir(self):
        """Dynamic output directory based on model ID and mode."""
        model_name_clean = self.model_id.replace("/", "_")
        mode = "ensemble" if self.use_ensemble else self.mode
        
        # Append density config to output dir
        density_suffix = f"_{self.density_column}"
        if not self.use_density: density_suffix = "_nodensity"
        if not self.normalize_density: density_suffix += "_raw"
        if not self.remove_outliers: density_suffix += "_noout"
        
        return f"{self.output_dir_base}/{model_name_clean}/{mode}{density_suffix}"

# Initialize Config
config = Config()
# config.use_ensemble = True # Uncomment to enable ensemble mode
print(f"Configuration initialized. Output directory: {config.output_dir}")

Configuration initialized. Output directory: models_knn_tox_refactored/answerdotai_ModernBERT-base/ensemble_nodensity_raw_noout


## 2. Data Processing

Encapsulates data loading, cleaning, density weight calculation, and tokenization.

In [4]:
class DataProcessor:
    """Handles data loading, preprocessing, and tokenization."""
    
    def __init__(self, config: Config, tokenizer):
        self.config = config
        self.tokenizer = tokenizer

    def load_and_preprocess(self, file_path: str) -> DatasetDict:
        """Loads CSV, cleans data, calculates density weights, and splits into train/test."""
        print(f"Loading dataset from {file_path}")
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")
            
        df = pd.read_csv(file_path)
        
        # Basic cleaning
        density_col = self.config.density_column
        if density_col not in df.columns:
            raise ValueError(f"'{density_col}' column missing in {file_path}. Available columns: {df.columns.tolist()}")
            
        # Ensure labels are binary integers
        if 'label' in df.columns and df['label'].dtype == 'object':
             df['label'] = df['label'].apply(lambda x: 1 if x == 'hate' else 0)
        
        if df[density_col].isna().any():
            print(f"Filling {df[density_col].isna().sum()} missing density values with median.")
            df[density_col] = df[density_col].fillna(df[density_col].median())
            
        # Remove outliers
        if self.config.remove_outliers:
            orig_len = len(df)
            lower = df[density_col].quantile(self.config.outlier_lower_q)
            upper = df[density_col].quantile(self.config.outlier_upper_q)
            df = df[(df[density_col] >= lower) & (df[density_col] <= upper)]
            print(f"Removed {orig_len - len(df)} outliers based on density.")

        if not self.config.use_density:
            df['density_weight'] = 1.0
        
        # Calculate density weights (normalized to [0.1, 2.0])
        if self.config.normalize_density:
            min_d, max_d = df[density_col].min(), df[density_col].max()
            if min_d == max_d:
                print("Warning: All density values are the same. Setting weights to 1.0")
                df['density_weight'] = 1.0
            else:
                df['density_weight'] = 0.1 + (df[density_col] - min_d) / (max_d - min_d) * 1.9
        else:
            print(f"Using raw {density_col} values as weights.")
            df['density_weight'] = df[density_col]
            
        # Split
        df_train = df.sample(frac=1-self.config.test_size, random_state=self.config.random_state)
        df_test = df.drop(df_train.index)
        
        # Convert to HF Dataset
        cols = ['text', 'label', 'density_weight']
        # Ensure columns exist
        cols = [c for c in cols if c in df.columns]
        
        train_ds = Dataset.from_pandas(df_train[cols])
        test_ds = Dataset.from_pandas(df_test[cols])
        
        # Cleanup index columns
        for ds in [train_ds, test_ds]:
            for col in ds.column_names:
                if col.startswith('__index'):
                    ds = ds.remove_columns(col)
                    
        return DatasetDict({"train": train_ds, "test": test_ds})

    def tokenize(self, batch):
        """Tokenization function to be used with dataset.map"""
        tokenized = self.tokenizer(
            batch['text'], 
            padding='max_length', 
            truncation=True, 
            max_length=self.config.max_length
        )
        # Add density weight if present
        if 'density_weight' in batch:
            tokenized['density_weight'] = batch['density_weight']
        
        # Rename label -> labels for Trainer
        if 'label' in batch:
            tokenized['labels'] = batch['label']
        elif 'labels' in batch:
            tokenized['labels'] = batch['labels']
            
        return tokenized

    def prepare_knn_datasets(self) -> Dict[str, DatasetDict]:
        """Process all KNN datasets defined in config."""
        processed = {}
        for name, path in self.config.knn_datasets.items():
            print(f"\nProcessing {name}...")
            try:
                ds_dict = self.load_and_preprocess(path)
                tokenized_ds = ds_dict.map(self.tokenize, batched=True)
                processed[name] = tokenized_ds
                print(f"Successfully processed {name}")
            except Exception as e:
                print(f"Error processing {name}: {e}")
        return processed
        
    def prepare_ensemble_datasets(self) -> Dict[str, DatasetDict]:
        """Process datasets for each identity term using dataHandler."""
        processed = {}
        print(f"Preparing ensemble datasets for terms: {self.config.identity_terms}")
        
        for term in self.config.identity_terms:
            print(f"\nProcessing term: {term}...")
            try:
                # Use dataHandler to get dataset
                dataset = dh.toxigenDataset(term, test_size=self.config.test_size)
                
                # Rename label column if needed
                if "label" in dataset["train"].features.keys():
                    dataset = dataset.rename_column("label", "labels")
                
                # Tokenize
                tokenized_ds = dataset.map(self.tokenize, batched=True)
                processed[term] = tokenized_ds
                print(f"Successfully processed {term}")
                
            except Exception as e:
                print(f"Error processing {term}: {e}")
                
        return processed

@dataclass
class DataCollatorWithDensityWeights(DataCollatorWithPadding):
    """Custom collator to preserve density_weight during batch creation."""
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        # Extract density weights if they exist
        density_weights = None
        if 'density_weight' in features[0]:
            density_weights = [f.pop('density_weight', 1.0) for f in features]
        
        # Remove 'text' field if it exists (Trainer expects only tensors)
        for f in features:
            f.pop('text', None) 
            
        batch = super().__call__(features)
        
        # Add weights back as tensor if they existed
        if density_weights is not None:
            batch['density_weight'] = torch.tensor(density_weights, dtype=torch.float32)
            
        return batch

## 3. Custom Trainer and Metrics

Defines the custom trainer that uses density weights for loss calculation.

In [5]:
class DensityWeightedTrainer(Trainer):
    """Trainer that uses density weights in loss calculation."""
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        density_weights = inputs.pop("density_weight", None)
        labels = inputs.pop("labels", None)
        
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        if labels is not None:
            loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
            loss_per_sample = loss_fct(logits, labels)
            
            if density_weights is not None:
                density_weights = density_weights.to(loss_per_sample.device)
                loss = (loss_per_sample * density_weights).mean()
            else:
                loss = loss_per_sample.mean()
                
            outputs.loss = loss
            return (loss, outputs) if return_outputs else loss
            
        return (outputs.loss, outputs) if return_outputs else outputs.loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        """Save density weights during evaluation for metrics computation."""
        if not hasattr(self, "_saved_density_weights"):
            self._saved_density_weights = []
            
        if "density_weight" in inputs:
            dw = inputs["density_weight"]
            if torch.is_tensor(dw):
                dw = dw.detach().cpu().numpy()
            self._saved_density_weights.append(dw)
            
        return super().prediction_step(model, inputs, prediction_loss_only, ignore_keys)

    def evaluation_loop(self, dataloader, description, prediction_loss_only=None, ignore_keys=None, metric_key_prefix="eval"):
        output = super().evaluation_loop(dataloader, description, prediction_loss_only, ignore_keys, metric_key_prefix)
        
        # If we have saved weights, re-compute metrics with them
        if self.compute_metrics is not None and hasattr(self, "_saved_density_weights") and self._saved_density_weights:
            merged_dw = np.concatenate(self._saved_density_weights, axis=0)
            
            # Ensure lengths match (sometimes drop_last or padding might affect things, but usually fine in EvalLoop)
            if len(merged_dw) == len(output.label_ids):
                output.metrics.update(
                    self.compute_metrics(EvalPrediction(
                        predictions=output.predictions,
                        label_ids=output.label_ids,
                        inputs={"density_weight": merged_dw}
                    ))
                )
            else:
                print(f"Warning: Density weights length ({len(merged_dw)}) mismatch with labels ({len(output.label_ids)})")
                
            self._saved_density_weights = [] # Clear after use
            
        return output

def compute_metrics(eval_pred):
    """Compute weighted AUC and other metrics."""
    preds = eval_pred.predictions
    labels = eval_pred.label_ids
    inputs = getattr(eval_pred, "inputs", None)
    
    # Convert logits to probabilities
    if preds.ndim > 1:
        probs = np.exp(preds) / np.exp(preds).sum(-1, keepdims=True)
        probs = probs[:, 1]
    else:
        probs = 1 / (1 + np.exp(-preds))
        
    weights = None
    if inputs is not None and isinstance(inputs, dict) and "density_weight" in inputs:
        weights = inputs["density_weight"]
        
    try:
        if config.use_density:
            auc = roc_auc_score(labels, probs, sample_weight=weights)
        else:
            auc = roc_auc_score(labels, probs)
        f1 = f1_score(labels, np.argmax(preds, axis=1) if preds.ndim > 1 else (preds > 0.5).astype(int), average="weighted")
    except Exception as e:
        print(f"Error computing metrics: {e}")
        auc = 0.5
        f1 = 0.0
        
    return {"weighted_auc": auc, "f1": f1}

## 4. Training Manager

Encapsulates the training loop for both KNN and Ensemble modes.

In [6]:
class ModelTrainer:
    """Manages model creation and training."""
    
    def __init__(self, config: Config, tokenizer):
        self.config = config
        self.tokenizer = tokenizer
        
    def create_model(self):
        return AutoModelForSequenceClassification.from_pretrained(
            self.config.model_id,
            num_labels=self.config.num_labels,
            label2id=self.config.label2id,
            id2label=self.config.id2label
        )
        
    def train_knn(self, dataset_name: str, tokenized_dataset: DatasetDict):
        """Train a single model on a KNN dataset."""
        print(f"\n{'='*40}\nTraining KNN Model for {dataset_name}\n{'='*40}")
        
        output_dir = f"{self.config.output_dir}/{dataset_name}"
        return self._run_training(tokenized_dataset, output_dir)
        
    def train_ensemble(self, term: str, tokenized_dataset: DatasetDict):
        """Train a single model for an ensemble term."""
        print(f"\n{'='*40}\nTraining Ensemble Model for {term}\n{'='*40}")
        
        output_dir = f"{self.config.output_dir}/{term}"
        return self._run_training(tokenized_dataset, output_dir)

    def _run_training(self, dataset, output_dir):
        # Clear cache
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        model = self.create_model().to(device)
        
        args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=self.config.batch_size,
            per_device_eval_batch_size=self.config.batch_size,
            learning_rate=self.config.learning_rate,
            num_train_epochs=self.config.num_epochs,
            warmup_ratio=self.config.warmup_ratio,
            weight_decay=self.config.weight_decay,
            bf16=torch.cuda.is_available(),
            fp16=not torch.cuda.is_available() and hasattr(torch, 'has_mps'),
            logging_strategy="steps",
            logging_steps=self.config.logging_steps,
            eval_strategy="steps",
            eval_steps=self.config.eval_steps,
            save_strategy="steps",
            save_steps=self.config.save_steps,
            save_total_limit=self.config.save_limit,
            load_best_model_at_end=True,
            metric_for_best_model="weighted_auc",
            greater_is_better=True,
            report_to="none", 
            include_for_metrics=["inputs"] # Important for custom trainer
        )
        
        trainer = DensityWeightedTrainer(
            model=model,
            args=args,
            train_dataset=dataset["train"],
            eval_dataset=dataset["test"],
            compute_metrics=compute_metrics,
            data_collator=DataCollatorWithDensityWeights(self.tokenizer),
            callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
        )
        
        trainer.train()
        
        # Save final model
        final_path = f"{output_dir}/final"
        model.save_pretrained(final_path)
        self.tokenizer.save_pretrained(f"{output_dir}/tokenizer")
        print(f"Saved model to {final_path}")
        
        return final_path

## 5. Evaluation and Ensembles

Classes and functions for evaluating trained models, including Ensembles, MFT (CheckList) and Confusion Matrices.

In [7]:
class DensityWeightedEnsemble:
    """Density-weighted ensemble classifier."""
    def __init__(self, model_paths, density_weights_path, use_median=False):
        self.model_paths = model_paths
        self.models = {}
        self.tokenizers = {}
        self.pipelines = {}
        
        # Load density weights from CSV
        if os.path.exists(density_weights_path):
            df_densities = pd.read_csv(density_weights_path)
            print(f"Loaded density weights from {density_weights_path}")
            
            weight_column = 'median' if use_median else 'average'
            self.weights = {}
            for _, row in df_densities.iterrows():
                term = row['term']
                weight = row[weight_column]
                self.weights[term] = weight
                
            # Only keep weights for terms that have models
            self.term_names = [term for term in model_paths.keys() if term in self.weights]
            self.weights = {term: self.weights[term] for term in self.term_names}
            
            # Normalize weights
            weight_sum = sum(self.weights.values())
            self.weights = {k: v/weight_sum for k, v in self.weights.items()}
            print(f"Normalized density weights: {self.weights}")
        else:
            print(f"Warning: Density weights file {density_weights_path} not found. Using uniform weights.")
            self.term_names = list(model_paths.keys())
            self.weights = {term: 1.0/len(self.term_names) for term in self.term_names}
        
        # Load models
        for term in self.term_names:
            paths = model_paths[term]
            print(f"Loading model for {term}...")
            self.tokenizers[term] = AutoTokenizer.from_pretrained(paths['tokenizer_path'])
            self.models[term] = AutoModelForSequenceClassification.from_pretrained(paths['model_path']).to(device)

    def predict(self, texts):
        if not isinstance(texts, list):
            texts = [texts]
        
        batch_size = 32
        all_model_scores = {term: [] for term in self.term_names}
        
        for term in self.term_names:
            model = self.models[term]
            tokenizer = self.tokenizers[term]
            model_scores = []
            
            for i in range(0, len(texts), batch_size):
                batch_texts = texts[i:i+batch_size]
                inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt", max_length=140)
                inputs = {k: v.to(device) for k, v in inputs.items()}
                
                with torch.no_grad():
                    outputs = model(**inputs)
                    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
                    model_scores.extend(probs[:, 1].cpu().numpy())
            
            all_model_scores[term] = np.array(model_scores)
            
        # Weighted average
        model_scores_array = np.array([all_model_scores[term] for term in self.term_names])
        weights_array = np.array([self.weights[term] for term in self.term_names])
        weighted_scores = np.sum(model_scores_array * weights_array[:, np.newaxis], axis=0)
        
        hate_predictions = weighted_scores >= 0.5
        final_predictions = ["hate" if pred else "no hate" for pred in hate_predictions]
        return final_predictions, weighted_scores.tolist()

class EnsembleClassifier:
    """Standard ensemble classifier (Majority Vote or Weighted Average)."""
    def __init__(self, model_paths, voting="majority", weights=None):
        self.model_paths = model_paths
        self.voting = voting
        self.models = {}
        self.tokenizers = {}
        self.term_names = list(model_paths.keys())
        
        if weights is None:
            self.weights = {term: 1.0/len(model_paths) for term in model_paths.keys()}
        else:
            self.weights = weights
            weight_sum = sum(self.weights.values())
            self.weights = {k: v/weight_sum for k, v in self.weights.items()}
            
        for term, paths in model_paths.items():
            print(f"Loading model for {term}...")
            self.tokenizers[term] = AutoTokenizer.from_pretrained(paths['tokenizer_path'])
            self.models[term] = AutoModelForSequenceClassification.from_pretrained(paths['model_path']).to(device)

    def predict(self, texts):
        if not isinstance(texts, list):
            texts = [texts]
            
        batch_size = 32
        all_model_scores = {term: [] for term in self.term_names}
        
        for term in self.term_names:
            model = self.models[term]
            tokenizer = self.tokenizers[term]
            model_scores = []
            
            for i in range(0, len(texts), batch_size):
                batch_texts = texts[i:i+batch_size]
                inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt", max_length=140)
                inputs = {k: v.to(device) for k, v in inputs.items()}
                
                with torch.no_grad():
                    outputs = model(**inputs)
                    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
                    model_scores.extend(probs[:, 1].cpu().numpy())
            
            all_model_scores[term] = np.array(model_scores)
            
        model_scores_array = np.array([all_model_scores[term] for term in self.term_names])
        
        if self.voting == "majority":
            binary_preds = (model_scores_array >= 0.5).astype(int)
            vote_counts = np.sum(binary_preds, axis=0)
            hate_predictions = vote_counts > (len(self.term_names) / 2)
            confidences = vote_counts / len(self.term_names)
        elif self.voting == "weighted_average":
            weights_array = np.array([self.weights[term] for term in self.term_names])
            weighted_scores = np.sum(model_scores_array * weights_array[:, np.newaxis], axis=0)
            hate_predictions = weighted_scores >= 0.5
            confidences = weighted_scores
            
        final_predictions = ["hate" if pred else "no hate" for pred in hate_predictions]
        return final_predictions, confidences.tolist()

class Evaluator:
    """Handles model evaluation, including cross-term analysis."""
    
    def __init__(self, config: Config):
        self.config = config
        
    def evaluate_direct(self, model_path, tokenizer_path, test_dataset):
        """Direct evaluation using manual batch processing."""
        print(f"Evaluating {model_path}...")
        try:
            tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
            model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
            model.eval()
            
            preds = []
            probs = []
            # Handle label/labels column mismatch
            if 'label' in test_dataset:
                label_col = 'label'
            elif 'labels' in test_dataset:
                label_col = 'labels'
            else:
                print(f"Error: No label column found in dataset. Available columns: {test_dataset.columns}")
                return None
            
            true_labels = test_dataset[label_col].tolist() if hasattr(test_dataset[label_col], 'tolist') else test_dataset[label_col]
            texts = test_dataset['text'].tolist() if hasattr(test_dataset['text'], 'tolist') else test_dataset['text']
            
            batch_size = 32
            for i in range(0, len(texts), batch_size):
                batch_texts = texts[i:i+batch_size]
                inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt", max_length=self.config.max_length).to(device)
                with torch.no_grad():
                    outputs = model(**inputs)
                    batch_probs = torch.nn.functional.softmax(outputs.logits, dim=1)
                preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
                probs.extend(batch_probs[:, 1].cpu().numpy())
                
            acc = accuracy_score(true_labels, preds)
            bal_acc = balanced_accuracy_score(true_labels, preds)
            prec, rec, f1, _ = precision_recall_fscore_support(true_labels, preds, average='binary')
            try:
                auc = roc_auc_score(true_labels, probs)
            except:
                auc = 0.5
                
            results = {
                "Accuracy": acc, "Balanced Accuracy": bal_acc,
                "Precision": prec, "Recall": rec, "F1": f1, "AUC": auc
            }
            print(f"Results: {results}")
            self.plot_confusion_matrix(true_labels, preds, title=f"Confusion Matrix: {model_path.split('/')[-2]}")
            return results
        except Exception as e:
            print(f"Error evaluating {model_path}: {e}")
            return None

    def evaluate_ensemble(self, ensemble, test_dataset, name="Ensemble"):
        """Evaluate an ensemble model."""
        print(f"Evaluating {name}...")
        texts = test_dataset['text'].tolist() if hasattr(test_dataset['text'], 'tolist') else test_dataset['text']
        true_labels = test_dataset['label'].tolist() if hasattr(test_dataset['label'], 'tolist') else test_dataset['label']
        
        preds_str, confidences = ensemble.predict(texts)
        preds = [1 if p == 'hate' else 0 for p in preds_str]
        
        acc = accuracy_score(true_labels, preds)
        f1 = f1_score(true_labels, preds, average='binary')
        try:
            auc = roc_auc_score(true_labels, confidences)
        except:
            auc = 0.5
            
        results = {"Accuracy": acc, "F1": f1, "AUC": auc}
        print(f"{name} Results: {results}")
        self.plot_confusion_matrix(true_labels, preds, title=f"Confusion Matrix: {name}")
        return results

    def plot_confusion_matrix(self, y_true, y_pred, title="Confusion Matrix"):
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title(title)
        plt.show()

    def plot_comparison(self, results: Dict[str, Dict[str, float]]):
        if not results:
            return
        df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "Model"})
        df_melted = df.melt(id_vars="Model", var_name="Metric", value_name="Score")
        plt.figure(figsize=(12, 6))
        sns.barplot(data=df_melted, x="Model", y="Score", hue="Metric")
        plt.title("Model Comparison")
        plt.ylim(0, 1.0)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()
        
    def plot_radar_comparison(self, results: Dict[str, Dict[str, float]]):
        """Plots a radar chart for model comparison."""
        if not results:
            return
            
        df = pd.DataFrame(results).T
        categories = list(df.columns)
        N = len(categories)
        angles = [n / float(N) * 2 * np.pi for n in range(N)]
        angles += angles[:1]
        
        plt.figure(figsize=(10, 10))
        ax = plt.subplot(111, polar=True)
        plt.xticks(angles[:-1], categories)
        
        for model_name, metrics in results.items():
            values = list(metrics.values())
            values += values[:1]
            ax.plot(angles, values, linewidth=1, linestyle='solid', label=model_name)
            ax.fill(angles, values, alpha=0.1)
            
        plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
        plt.title("Model Performance Radar")
        plt.show()

## 6. Main Execution

Orchestrates the entire workflow based on configuration.

In [8]:
# 1. Initialize Config & Tokenizer
config = Config()
# Toggle this to switch between KNN and Ensemble modes
# config.use_ensemble = True 

tokenizer = AutoTokenizer.from_pretrained(config.model_id)
processor = DataProcessor(config, tokenizer)
trainer_manager = ModelTrainer(config, tokenizer)
evaluator = Evaluator(config)

trained_models = {}

if config.use_ensemble:
    print("STARTING ENSEMBLE TRAINING PIPELINE")
    # Prepare Data
    processed_datasets = processor.prepare_ensemble_datasets()
    
    if not processed_datasets:
        print("No datasets processed for ensemble.")
    else:
        # Train Models
        for term, ds in processed_datasets.items():
            path = trainer_manager.train_ensemble(term, ds)
            trained_models[term] = path
            
else:
    print("STARTING KNN TRAINING PIPELINE")
    # Prepare Data
    processed_datasets = processor.prepare_knn_datasets()
    
    if not processed_datasets:
        print("No datasets processed for KNN.")
    else:
        # Train Models
        for name, ds in processed_datasets.items():
            path = trainer_manager.train_knn(name, ds)
            trained_models[name] = path

print(f"\nTraining completed. Models saved in {config.output_dir}")

STARTING ENSEMBLE TRAINING PIPELINE
Preparing ensemble datasets for terms: ['asian', 'black', 'chinese', 'jewish', 'latino', 'lgbtq', 'mental_dis', 'mexican', 'middle_east', 'muslim', 'native_american', 'physical_dis', 'women']

Processing term: asian...


Map:   0%|          | 0/15907 [00:00<?, ? examples/s]

Map:   0%|          | 0/3977 [00:00<?, ? examples/s]

Successfully processed asian

Processing term: black...


Map:   0%|          | 0/15902 [00:00<?, ? examples/s]

...

Step,Training Loss,Validation Loss,Weighted Auc,F1
1000,0.085600,0.345756,0.957206,0.903691


Saved model to models_knn_tox_refactored/answerdotai_ModernBERT-base/ensemble_nodensity_raw_noout/physical_dis/final

Training Ensemble Model for women


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,Weighted Auc,F1
1000,0.059300,0.322090,0.957546,0.913217


Saved model to models_knn_tox_refactored/answerdotai_ModernBERT-base/ensemble_nodensity_raw_noout/women/final

Training completed. Models saved in models_knn_tox_refactored/answerdotai_ModernBERT-base/ensemble_nodensity_raw_noout
